# llm-cache-router — Interactive Playground

Семантический кэш, мульти-провайдер роутинг и cost tracking в одной async-библиотеке.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/svalench/llm-cache-router/blob/main/notebooks/playground.ipynb)

**Что попробуем:**
1. Установка пакета
2. Два похожих запроса → `cache_hit=False` затем `cache_hit=True`
3. Streaming и `router.stats()`

Для production-стека с Redis/Qdrant см. [docker-compose в репозитории](https://github.com/svalench/llm-cache-router#try-in-one-command).

In [ ]:
!pip install -q llm-cache-router

In [ ]:
import os
from getpass import getpass

api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    api_key = getpass("OpenAI API key: ")
    os.environ["OPENAI_API_KEY"] = api_key

In [ ]:
from llm_cache_router import CacheConfig, LLMRouter, RoutingStrategy

router = LLMRouter(
    providers={
        "openai": {
            "api_key": api_key,
            "models": ["gpt-4o-mini"],
        },
    },
    cache=CacheConfig(
        backend="memory",
        threshold=0.92,
        ttl=3600,
        max_entries=1000,
    ),
    strategy=RoutingStrategy.CHEAPEST_FIRST,
    budget={"daily_usd": 5.0},
)

print("Router ready (memory cache backend)")

In [ ]:
async def demo_cache_hit() -> None:
    q1 = "What is a semantic cache?"
    q2 = "Explain semantic caching for LLMs in one sentence."

    r1 = await router.complete(
        messages=[{"role": "user", "content": q1}],
        model="gpt-4o-mini",
    )
    print("--- Query 1 ---")
    print(r1.content[:300], "..." if len(r1.content) > 300 else "")
    print(f"cache_hit={r1.cache_hit}  cost=${r1.cost_usd:.6f}")

    r2 = await router.complete(
        messages=[{"role": "user", "content": q2}],
        model="gpt-4o-mini",
    )
    print("\n--- Query 2 (similar) ---")
    print(r2.content[:300], "..." if len(r2.content) > 300 else "")
    print(f"cache_hit={r2.cache_hit}  cost=${r2.cost_usd:.6f}")

await demo_cache_hit()

In [ ]:
stats = router.stats()
print(f"cache_hit_rate={stats.cache_hit_rate:.0%}")
print(f"total_requests={stats.total_requests}")
print(f"saved_cost_usd=${stats.saved_cost_usd:.6f}")
stats

In [ ]:
async def demo_stream() -> None:
    print("Streaming response:")
    async for chunk in router.stream(
        messages=[{"role": "user", "content": "Say hello in three words."}],
        model="gpt-4o-mini",
    ):
        if chunk.content:
            print(chunk.content, end="", flush=True)
    print()

await demo_stream()

## Next steps

- **Docker demo** (Redis + Qdrant + FastAPI): `docker compose up --build` — [инструкция](https://github.com/svalench/llm-cache-router/tree/main/examples/demo)
- **Документация**: [README](https://github.com/svalench/llm-cache-router#readme)
- **PyPI**: [llm-cache-router](https://pypi.org/project/llm-cache-router/)